# 03 — Conectar busca e geração: o primeiro RAG

As etapas 01 e 02 prepararam documentos rastreáveis e um índice persistente. Agora uma pergunta percorre os dois componentes: o encoder encontra trechos; o Qwen recebe a pergunta junto com esses trechos e redige a resposta. Esse padrão é RAG (geração aumentada por recuperação).

```mermaid
flowchart LR
 A[pergunta] --> B[encoder de consulta]
 B --> C[Chroma / HNSW]
 C --> D[trechos + páginas]
 A --> E[prompt]
 D --> E
 E --> F[Qwen 3B]
 F --> G[resposta + fontes]
```

O Qwen continua sendo o mesmo modelo da etapa 00. A diferença controlada é a evidência adicionada ao prompt.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: 03 — RAG com fontes e Qwen
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

## 1. Recarregar os componentes persistidos

Este notebook pode começar em um kernel novo: ele reabre o Chroma da etapa 02, carrega o mesmo encoder da etapa 01 e carrega a LLM definida na etapa 00. O encoder pode ficar na CPU para reservar VRAM ao Qwen. Ainda não há adaptador QLoRA; primeiro será medido o comportamento da base com e sem recuperação.

Se o manifesto, coleção ou modelo não corresponder ao esperado, interrompa e corrija a etapa anterior em vez de misturar artefatos incompatíveis.

---

### Três componentes, três funções

O encoder transforma texto em vetores para localizar passagens; o Chroma armazena e busca essas passagens; o Qwen redige a resposta. Exemplo: para “como funciona a atenção?”, o encoder representa a pergunta, o banco retorna trechos sobre Q/K/V, e o Qwen explica usando esses trechos. O encoder não escreve a resposta e o Qwen não executa a busca por conta própria.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import chromadb
from sentence_transformers import SentenceTransformer
from rag import ENCODER, COLECAO, carregar_llm, recuperar, montar_mensagens, gerar

cliente = chromadb.PersistentClient(path=str(DIRETORIO_PROJETO / "banco_chroma"))
colecao = cliente.get_collection(COLECAO)
encoder = SentenceTransformer(ENCODER, device="cpu")
modelo, tokenizador = carregar_llm()
print("Registros disponíveis:", colecao.count())

## 2. Buscar e ler os trechos antes de gerar

A pergunta é convertida em embedding e usada para consultar o Chroma. Antes de chamar a LLM, examine títulos, páginas, distâncias e conteúdo dos resultados. Essa inspeção separa dois tipos de problema: o índice pode não recuperar a evidência necessária, ou o Qwen pode receber a evidência e ainda interpretá-la mal.

A distância serve para ordenar candidatos, não para declarar que um trecho é verdadeiro. Confirme se cada resultado realmente responde a uma parte da pergunta.

---

### Como avaliar o resultado da busca

Suponha que a pergunta peça (1) a equação da atenção e (2) um exemplo PyTorch. Um chunk que explica a equação cobre a primeira necessidade; se nenhum resultado trouxer código, a recuperação está incompleta mesmo que o primeiro trecho tenha distância pequena. A inspeção de conteúdo e localização é necessária porque distância ordena candidatos, mas não atesta cobertura nem verdade.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
pergunta_rag = "Como a atenção pode representar sequências e quais cuidados existem na previsão de inadimplência de consumidores?"
trechos = recuperar(colecao, encoder, pergunta_rag, usar_mmr=True, k=4, fetch_k=12)
for trecho in trechos:
    print(trecho["source_id"], "página", trecho["page"])
    print(trecho["texto"][:500], "\n")

## 3. Gerar e conferir a resposta com evidências

Agora a pergunta e os trechos recuperados são organizados em mensagens. O prompt orienta o Qwen a usar o contexto e citar as fontes disponíveis. Instruções ajudam, mas não garantem que todas as afirmações fiquem apoiadas; por isso as citações devem ser conferidas no documento e na página indicados.

O orçamento de contexto é finito: sistema, histórico, pergunta, trechos e resposta competem pelos tokens. Se os trechos excederem o limite, reduza-os com uma regra explícita e inspecionável. Não corte conteúdo silenciosamente.

---

### O mecanismo do RAG

RAG significa geração aumentada por recuperação. O texto recuperado é concatenado ao prompt como contexto; os pesos do Qwen não mudam. Exemplo de fluxo: pergunta → dois trechos com páginas → prompt com pergunta e trechos → resposta que cita essas páginas. O resultado depende de duas etapas: recuperar a evidência adequada e o gerador usá-la fielmente.

### Orçamento de contexto: conta ilustrativa

Se a janela disponível for 1.024 tokens, uma divisão possível é: instrução 150 + histórico 200 + pergunta 50 + evidências 400 + reserva para resposta 224 = 1.024. Se as evidências crescerem para 600, a soma vira 1.224 e excede o limite. Será necessário reduzir histórico ou selecionar trechos; truncar sem inspeção pode apagar a frase que sustentava a resposta.

Uma citação útil aponta para a origem efetivamente recuperada, por exemplo título do documento + página PDF + trecho. Citar um documento apenas relacionado ao tema não fundamenta uma afirmação.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Parte IV — RAG e interface web
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from rag import gerar
mensagens_rag = montar_mensagens(pergunta_rag, trechos)
print(mensagens_rag[-1]["content"])
resposta_rag = gerar(modelo, tokenizador, mensagens_rag)
print(resposta_rag)

## 4. Levar o mesmo fluxo à interface

O notebook demonstrou manualmente busca → contexto → geração. A interface web chama essa lógica para cada mensagem e apresenta fontes junto à resposta. Assim, o usuário não precisa abrir o notebook para consultar o corpus. O próximo passo de modelagem será estudar QLoRA sobre a mesma base; depois, a etapa 05 comparará o efeito do ajuste e do RAG.

A interface deve carregar uma configuração por vez para respeitar a VRAM. O modo ajustado só pode ser selecionado quando o adaptador da etapa 04 existir e tiver sido avaliado.

---

### Do notebook à aplicação

No notebook, cada transformação foi inspecionada separadamente. A interface chama o mesmo fluxo para cada pergunta: recebe a mensagem, envia à API, recupera evidências e apresenta resposta e fontes. A etapa 04 produz um adaptador opcional; a etapa 05 decide, com avaliação, se ele deve ser oferecido no chat.